# 03 - Deep Learning Segmentation Model

This notebook implements the deep learning approach for dendrite segmentation in SEM images.

A YOLO segmentation model is trained using the labeled dataset exported from Roboflow.  
The dataset contains polygon annotations of dendrite regions and is organized into **train / validation / test** splits.

Because the Roboflow free tier does not allow downloading the trained model weights (`best.pt`), the model is trained locally using the **Ultralytics YOLO framework** in order to obtain the weights required for inference and evaluation.

The objectives of this notebook are:

- Train a YOLO segmentation model on the dendrite dataset
- Save the trained model weights (`best.pt`)
- Run inference on the test images
- Generate predicted segmentation masks

The generated predictions will later be compared against:

- Ground Truth annotations  
- The Classical Segmentation Pipeline results  

Evaluation metrics used for comparison include:

- IoU (Intersection over Union)
- Dice coefficient
- Precision
- Recall

This enables a quantitative comparison between **classical image processing methods** and **deep learning based segmentation** for dendrite detection.

### Dataset Structure

The dataset is organized in YOLO segmentation format and contains
three splits:

- **train** - images used for training
- **valid** - images used for validation
- **test** - images used for evaluation

Each image has a corresponding label file containing polygon annotations
that describe the dendritic structures.

In [4]:
from pathlib import Path

DATASET_ROOT = Path("dataset/SEM_Dendrite_Segmentation_v2.v3i.yolov11")
DATA_YAML = DATASET_ROOT / "data.yaml"

print("Dataset root exists:", DATASET_ROOT.exists())
print("data.yaml exists:", DATA_YAML.exists())

Dataset root exists: True
data.yaml exists: True


### Training Configuration

The model is trained using the following parameters:

- **Epochs:** 50
- **Image size:** 640 × 640
- **Batch size:** 4
- **Task:** Instance segmentation

During training, the network learns to predict segmentation masks
for dendritic structures in SEM images.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=4,
    project="runs_local",
    name="dendrite_seg",
    pretrained=True
)

Ultralytics 8.4.21  Python-3.12.7 torch-2.10.0+cpu CPU (12th Gen Intel Core i7-1255U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset\SEM_Dendrite_Segmentation_v2.v3i.yolov11\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dendrite_seg, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ove

### Training2 Configuration

The final deep learning model was trained using the following parameters:

- **Model:** YOLOv11s-seg
- **Epochs:** 150
- **Image size:** 640 × 640
- **Batch size:** 4
- **Task:** Instance segmentation
- **Initialization:** Pretrained weights (transfer learning)

During training, the network learns to predict segmentation masks for dendritic structures in SEM images.

This configuration provides higher model capacity compared to the initial YOLOv11n model and improves segmentation performance on complex dendritic structures.

In [8]:
from ultralytics import YOLO

model = YOLO("yolo11s-seg.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=150,
    imgsz=640,
    batch=4,
    device="cpu",
    augment=True,
    project="runs_local",
    name="dendrite_seg_s"
)

Ultralytics 8.4.21  Python-3.12.7 torch-2.10.0+cpu CPU (12th Gen Intel Core i7-1255U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset\SEM_Dendrite_Segmentation_v2.v3i.yolov11\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dendrite_seg_s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, o

### Training Results

Two training configurations were tested during the deep learning stage.

An initial model based on **YOLOv11n-seg** was trained as a baseline.  
While this lightweight model successfully detected some dendritic structures, its segmentation accuracy was limited due to the small model capacity and the complexity of the SEM images.

To improve performance, a second model using **YOLOv11s-seg** was trained for more epochs.  
This configuration achieved higher Precision, Recall, and mAP scores and produced more stable segmentation masks.

Based on these results, **YOLOv11s-seg was selected as the final deep learning model for the project**.

### Training Output

After training is completed, the best model weights are saved locally as:

`runs_local/dendrite_seg/weights/best.pt`

These weights represent the trained segmentation model
and will be used for inference and evaluation.

In [ ]:
from ultralytics import YOLO

BEST_MODEL = Path("runs/segment/runs_local/dendrite_seg_s/weights/best.pt")

print("Best model exists:", BEST_MODEL.exists())
print(BEST_MODEL)

best_model = YOLO(str(BEST_MODEL))

Best model exists: True
runs\segment\runs_local\dendrite_seg_s\weights\best.pt


In [42]:
pred_easy = best_model.predict(
    source="datasets/for_annotation/easy",
    save=True,
    project="runs_local",
    name="predict_easy",
    imgsz=640,
    conf=0.2,
    iou=0.5,
    retina_masks=True,
    show_labels=False,
    show_conf=False,
    show_boxes=False
)


image 1/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\easy\Ag_1e-8_007.png: 448x640 62 dendritess, 469.5ms
image 2/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\easy\Ag_1e-8_008.png: 448x640 50 dendritess, 454.9ms
image 3/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\easy\Ag_1e-8_009.png: 448x640 76 dendritess, 1915.2ms
image 4/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\easy\Ag_1e-8_010.png: 448x640 248 dendritess, 309.7ms
image 5/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\easy\Ag_1e-8_011.png: 448x640 300 dendritess, 222.4ms
image 6/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\easy\Ag_1e-8_012.png: 448x640 40 dendritess, 421.4ms
image 7/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\easy\Ag_1e-8_017.png: 448x640 298 dendritess, 336.2ms
Speed: 26.5ms preprocess, 589.9ms inference, 1779.4ms postprocess per i

In [43]:
pred_hard = best_model.predict(
    source="datasets/for_annotation/hard",
    save=True,
    project="runs_local",
    name="predict_hard",
    imgsz=640,
    conf=0.2,
    iou=0.5,
    retina_masks=True,
    show_labels=False,
    show_conf=False,
    show_boxes=False
)


image 1/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\hard\70nm_R_50nm_pitch_ETD_001.png: 448x640 41 dendritess, 381.6ms
image 2/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\hard\70nm_R_50nm_pitch_ETD_002.png: 448x640 135 dendritess, 294.3ms
image 3/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\hard\70nm_R_50nm_pitch_ETD_003.png: 448x640 164 dendritess, 251.4ms
image 4/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\hard\70nm_R_50nm_pitch_ETD_004.png: 448x640 76 dendritess, 271.4ms
image 5/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\hard\70nm_R_50nm_pitch_ETD_005.png: 448x640 30 dendritess, 221.0ms
image 6/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\hard\70nm_R_50nm_pitch_ETD_006.png: 448x640 29 dendritess, 225.0ms
image 7/7 c:\Users\sara tuvian\Desktop\   \ \notebooks\datasets\for_annotation\hard\ETD_097.png: 448x640 17 dendritess, 2

### Model Performance Summary

Two YOLO segmentation models were trained during the development of the deep learning pipeline.

The first model used the **YOLO11n-seg architecture** and was trained for **50 epochs**. While the model was able to detect some dendritic structures, the segmentation quality and detection consistency were limited due to the small model capacity and short training duration.

To improve performance, a second model based on **YOLO11s-seg** was trained for **150 epochs**. This model has a larger architecture and allows better feature learning from the dataset.

After training, the best model from the second experiment was applied to images from both the **EASY** and **HARD** subsets in order to evaluate its ability to detect dendritic structures.

Visual inspection of the predictions shows that the model is able to detect many dendritic branches across the images. In the EASY images, the detected masks generally follow the main dendritic structures well. In the HARD images, detection becomes more challenging due to higher noise levels and the presence of thin, low-contrast dendrites.

Although some **false detections** and **missed structures** remain, the model is still able to identify a significant portion of the dendritic patterns.

These results demonstrate that deep learning–based instance segmentation can successfully detect dendritic structures in SEM images, even when the training dataset is relatively small. Further improvements could likely be achieved by expanding the dataset, refining annotations, and performing additional training or hyperparameter tuning.

## Export YOLO Binary Masks

The following cell extracts binary segmentation masks from the YOLO prediction results and saves them to disk.

For each image, all predicted instance masks are merged into a single binary mask.

The masks are saved separately for the **EASY** and **HARD** subsets under:

- `runs/segment/runs_local/yolo_binary_masks/easy`
- `runs/segment/runs_local/yolo_binary_masks/hard`

In [44]:
from pathlib import Path
import numpy as np
import cv2

# --------------------------------------------------
# Output folders
# --------------------------------------------------
BASE_OUT = Path("runs/segment/runs_local/yolo_binary_masks")
EASY_OUT = BASE_OUT / "easy"
HARD_OUT = BASE_OUT / "hard"

EASY_OUT.mkdir(parents=True, exist_ok=True)
HARD_OUT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Helper function
# --------------------------------------------------
def export_combined_masks(pred_results, out_dir: Path, threshold: float = 0.5):
    saved = 0

    for r in pred_results:
        img_path = Path(r.path)
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Skipping unreadable image: {img_path}")
            continue

        h, w = img.shape[:2]
        combined_mask = np.zeros((h, w), dtype=np.uint8)

        if r.masks is not None:
            masks = r.masks.data.cpu().numpy()

            for m in masks:
                m_bin = (m > threshold).astype(np.uint8)
                m_bin = cv2.resize(m_bin, (w, h), interpolation=cv2.INTER_NEAREST)
                combined_mask = np.maximum(combined_mask, m_bin)

        out_path = out_dir / f"{img_path.stem}_yolo_mask.png"
        cv2.imwrite(str(out_path), combined_mask * 255)
        saved += 1

    return saved

# --------------------------------------------------
# Export masks
# --------------------------------------------------
easy_saved = export_combined_masks(pred_easy, EASY_OUT)
hard_saved = export_combined_masks(pred_hard, HARD_OUT)

print(f"EASY masks saved: {easy_saved} -> {EASY_OUT}")
print(f"HARD masks saved: {hard_saved} -> {HARD_OUT}")

EASY masks saved: 7 -> runs\segment\runs_local\yolo_binary_masks\easy
HARD masks saved: 7 -> runs\segment\runs_local\yolo_binary_masks\hard


### Training Metrics

During training, YOLO logs performance metrics for each epoch.
The following table shows the training progress recorded in `results.csv`.

Key metrics include:

- **Precision** - proportion of correct detections
- **Recall** - proportion of detected dendrites
- **mAP50** - detection accuracy at IoU threshold 0.5
- **Segmentation mAP** - segmentation quality across IoU thresholds

These metrics help evaluate whether the model improves during training.

In [45]:
import pandas as pd

RESULTS_PATH = Path("runs/segment/runs_local/dendrite_seg_s/results.csv")

results_df = pd.read_csv(RESULTS_PATH)

results_df.head()

,epoch,time,train/box_loss,train/seg_loss,train/cls_loss,train/dfl_loss,train/sem_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),...,metrics/mAP50(M),metrics/mAP50-95(M),val/box_loss,val/seg_loss,val/cls_loss,val/dfl_loss,val/sem_loss,lr/pg0,lr/pg1,lr/pg2
0,1,22.6424,2.97863,5.11718,2.94694,1.70682,0,0.10851,0.07143,0.05363,...,0.04966,0.01461,2.97678,4.47107,2.41709,1.32099,0,0.000040,0.000040,0.000040
1,2,46.7866,2.47742,4.17000,2.48944,1.38556,0,0.22099,0.08472,0.08924,...,0.07823,0.02523,2.63671,3.90545,3.99390,1.24690,0,0.000099,0.000099,0.000099
2,3,71.0072,2.21248,3.71496,1.91360,1.21222,0,0.16990,0.09635,0.09277,...,0.08670,0.02549,2.48052,3.86546,5.23019,1.19923,0,0.000158,0.000158,0.000158
3,4,95.0173,1.95763,3.60820,1.88153,1.20560,0,0.15896,0.10963,0.09082,...,0.09090,0.02938,2.37882,3.84394,3.25326,1.15284,0,0.000216,0.000216,0.000216
4,5,119.3940,2.14561,3.30938,1.73218,1.17098,0,0.22015,0.10299,0.10310,...,0.08732,0.02792,2.36666,3.69846,3.09547,1.13495,0,0.000273,0.000273,0.000273


In [46]:
results_df.tail(1)

,epoch,time,train/box_loss,train/seg_loss,train/cls_loss,train/dfl_loss,train/sem_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),...,metrics/mAP50(M),metrics/mAP50-95(M),val/box_loss,val/seg_loss,val/cls_loss,val/dfl_loss,val/sem_loss,lr/pg0,lr/pg1,lr/pg2
149,150,3561.29,1.60141,2.34072,0.98705,0.93227,0,0.34078,0.2309,0.22228,...,0.21991,0.09326,2.30443,2.75868,1.35745,0.97174,0,0.000033,0.000033,0.000033


In [47]:
results_df[[
    "epoch",
    "metrics/precision(M)",
    "metrics/recall(M)",
    "metrics/mAP50(M)",
    "metrics/mAP50-95(M)"
]].tail()

,epoch,metrics/precision(M),metrics/recall(M),metrics/mAP50(M),metrics/mAP50-95(M)
145,146,0.35731,0.25083,0.21991,0.09326
146,147,0.35731,0.25083,0.21991,0.09326
147,148,0.35731,0.25083,0.21991,0.09326
148,149,0.35731,0.25083,0.21991,0.09326
149,150,0.35731,0.25083,0.21991,0.09326


### Training Results Analysis

The YOLO segmentation model was trained for **150 epochs** on the annotated SEM dendrite dataset using the **YOLO11s segmentation architecture**.

During training, the model progressively improved as the training losses decreased and the evaluation metrics stabilized toward the final epochs.

At the final stage of training, the model achieved the following **segmentation performance**:

- **Precision (Segmentation):** 0.357  
- **Recall (Segmentation):** 0.251  
- **mAP@0.5:** 0.220  
- **mAP@0.5:0.95:** 0.093  

These results indicate that the model successfully learned meaningful dendritic structures from the training data. Although the performance is moderate, the model is able to identify multiple dendritic branches across the images.

The relatively limited performance is expected due to several factors:

- the **small dataset size** (14 SEM images)
- the **high level of noise** present in SEM microscopy images
- the **complex branching morphology** of dendritic structures
- the presence of **thin and low-contrast dendrites** in the HARD subset

Despite these challenges, the trained model demonstrates the ability to detect dendritic regions and generate segmentation masks that follow the main dendritic structures.

### Conclusion

In this notebook, a **YOLO-based instance segmentation approach** was developed to detect dendritic structures in SEM microscopy images.

Two YOLO segmentation models were trained during the experimentation phase. The initial model used the **YOLO11n-seg architecture trained for 50 epochs**, while the improved model used the **YOLO11s-seg architecture trained for 150 epochs**, which provided better learning capacity and more stable results.

The final model was trained locally using the annotated dataset and evaluated on both the **Easy** and **Hard** image subsets.

The training metrics and prediction examples demonstrate that the model can learn meaningful dendritic patterns even from a relatively small dataset. The model is capable of identifying multiple dendritic branches within complex SEM images.

However, the segmentation quality is still influenced by noise and structural complexity, particularly in the **HARD images**, where dendrites are thinner and less distinct.

Overall, the results demonstrate that **deep learning–based segmentation using YOLO can detect dendritic structures in SEM images**, even with limited training data.

In the next stage of the project, the deep learning approach will be **compared with the classical image processing pipeline** developed earlier in order to evaluate the strengths and limitations of each method.